In [56]:
import pandas as pd
import numpy as np
from pathlib import Path


In [57]:
path=Path(r"D:\deep learning\Grad proj\model\kafka\weekly_with_total_lines")

In [58]:
path2=Path(r"D:\deep learning\Grad proj\model\author_weekly")

In [59]:
output_path = Path(r"D:\deep learning\Grad proj\model\kafka\g3")

In [60]:
output_path.mkdir(exist_ok=True)

# **Churn & Churn Acceleration**

In [61]:
def churn_rate(file):
    
    df=file.copy()
    df['churn_rate']=df['churn']/df['total_lines'].replace(0, pd.NA)
    
    return df

In [62]:
for file in path.glob("*.csv"):
    
    df=pd.read_csv(file)
    df=churn_rate(df)
    df.to_csv(file, index=False)
    print(f"{file.name} done")

week_01.csv done
week_02.csv done
week_03.csv done
week_04.csv done
week_05.csv done
week_06.csv done
week_07.csv done
week_08.csv done
week_09.csv done
week_10.csv done
week_11.csv done
week_12.csv done
week_13.csv done
week_14.csv done
week_15.csv done
week_16.csv done
week_17.csv done
week_18.csv done
week_19.csv done
week_20.csv done
week_21.csv done
week_22.csv done
week_23.csv done
week_24.csv done
week_25.csv done
week_26.csv done


In [63]:
weekly_data = {}

files = sorted(path.glob("week_*.csv"))

for i, file in enumerate(files):

    df = pd.read_csv(file)

    weekly_data[file.name] = df[
        ["artifact_id", "churn_rate", "commits"]
    ].copy()

    if i == 0:

        df["churn_acceleration"] = pd.NA
        df["commit_acceleration"] = pd.NA

    else:

        current_week = weekly_data[file.name]
        previous_week = weekly_data[files[i - 1].name]

        merged = current_week.merge(
            previous_week,
            on="artifact_id",
            how="left",
            suffixes=("_current", "_previous")
        )

        # If artifact did not exist in previous week,
        # consider its previous activity as 0
        previous_commits = merged["commits_previous"].fillna(0)
        previous_churn = merged["churn_rate_previous"].fillna(0)

        df["commit_acceleration"] = (
            merged["commits_current"] - previous_commits
        ).values

        df["churn_acceleration"] = (
            merged["churn_rate_current"] - previous_churn
        ).values

    # Save permanently
    df.to_csv(
        output_path / file.name,
        index=False
    )

    print(f"{file.name} done")

week_01.csv done


week_02.csv done
week_03.csv done
week_04.csv done
week_05.csv done
week_06.csv done
week_07.csv done
week_08.csv done
week_09.csv done
week_10.csv done
week_11.csv done
week_12.csv done
week_13.csv done
week_14.csv done
week_15.csv done
week_16.csv done
week_17.csv done
week_18.csv done
week_19.csv done
week_20.csv done
week_21.csv done
week_22.csv done
week_23.csv done
week_24.csv done
week_25.csv done
week_26.csv done


# **Churn & Commit Burstiness**

In [ ]:
files = sorted(output_path.glob("week_*.csv"))
data={}

for i,file in enumerate(files):
    
    df=pd.read_csv(file)
    data[file.name] = df[['artifact_id','commits']].copy()
    
    if i<3:
        df['commit_burstiness']=pd.NA
        
    else:
        current_week=data[file.name]
        previous_3weeks=data[files[i-3].name]
        previous_2weeks=data[files[i-2].name]   
        previous_1week=data[files[i-1].name]   
        
        previous_3weeks = previous_3weeks.rename(
            columns={"commits": "commits_w3"}
        )

        previous_2weeks = previous_2weeks.rename(
            columns={"commits": "commits_w2"}
        )

        previous_1week = previous_1week.rename(
            columns={"commits": "commits_w1"}
        )
        
        merged = previous_3weeks.merge(
                    previous_2weeks,
                    on="artifact_id",
                    how="outer"
                
                ).merge(previous_1week,
                                    on="artifact_id",
                                    how="outer")
                
        commit_values = merged[
            ["commits_w3", "commits_w2", "commits_w1"]
        ].fillna(0)        
                
        total_var=commit_values.var(axis=1,ddof=0)
        total_mean=commit_values.mean(axis=1)   
        
        merged["commit_burstiness"] = np.where(
            total_mean > 0,
            total_var / total_mean,
            0
        )

        burstiness = merged[
            ["artifact_id", "commit_burstiness"]
        ]

        df = df.drop(
            columns=["commit_burstiness"],
            errors="ignore"
        ).merge(
            burstiness,
            on="artifact_id",
            how="left"
        )
        
        
    df.to_csv(
        output_path / file.name,
        index=False
    )

    print(f"{file.name} done") 

week_01.csv done
week_02.csv done
week_03.csv done
week_04.csv done
week_05.csv done
week_06.csv done
week_07.csv done
week_08.csv done
week_09.csv done
week_10.csv done
week_11.csv done
week_12.csv done
week_13.csv done
week_14.csv done
week_15.csv done
week_16.csv done
week_17.csv done
week_18.csv done
week_19.csv done
week_20.csv done
week_21.csv done
week_22.csv done
week_23.csv done
week_24.csv done
week_25.csv done
week_26.csv done


In [67]:
files = sorted(output_path.glob("week_*.csv"))
data={}

for i,file in enumerate(files):
    
    df=pd.read_csv(file)
    data[file.name] = df[['artifact_id','churn']].copy()
    
    if i<3:
        df['churn_burstiness']=pd.NA
        
    else:
        current_week=data[file.name]
        previous_3weeks=data[files[i-3].name]
        previous_2weeks=data[files[i-2].name]   
        previous_1week=data[files[i-1].name]   
        
        previous_3weeks = previous_3weeks.rename(
            columns={"churn": "churns_w3"}
        )

        previous_2weeks = previous_2weeks.rename(
            columns={"churn": "churns_w2"}
        )

        previous_1week = previous_1week.rename(
            columns={"churn": "churns_w1"}
        )
        
        merged = previous_3weeks.merge(
                    previous_2weeks,
                    on="artifact_id",
                    how="outer"
                
                ).merge(previous_1week,
                                    on="artifact_id",
                                    how="outer")
                
        churn_values = merged[
    ["churns_w3", "churns_w2", "churns_w1"]
].fillna(0)
        total_std=churn_values.std(axis=1,ddof=0)
        total_mean=churn_values.mean(axis=1)   
        
        merged["churn_burstiness"] = np.where(
           total_std + total_mean > 0,
           (total_std - total_mean) / (total_std + total_mean),
           0
            )

        burstiness = merged[
            ["artifact_id", "churn_burstiness"]
        ]

        df = df.drop(
            columns=["churn_burstiness"],
            errors="ignore"
        ).merge(
            burstiness,
            on="artifact_id",
            how="left"
        )
        
        
    df.to_csv(
        output_path / file.name,
        index=False
    )

    print(f"{file.name} done") 

week_01.csv done
week_02.csv done
week_03.csv done
week_04.csv done
week_05.csv done
week_06.csv done
week_07.csv done
week_08.csv done
week_09.csv done
week_10.csv done
week_11.csv done
week_12.csv done
week_13.csv done
week_14.csv done
week_15.csv done
week_16.csv done
week_17.csv done
week_18.csv done
week_19.csv done
week_20.csv done
week_21.csv done
week_22.csv done
week_23.csv done
week_24.csv done
week_25.csv done
week_26.csv done
